# NB07 Summary — Ratio 5% + 10% Enhanced Report v8

Notebook này tổng hợp kết quả NB02–NB06 cho hai tỷ lệ labeled data `5%` và `10%`.

Notebook **không train** và **không cần load `.pt` checkpoint**. Các hình prediction được vẽ trực tiếp từ prediction JSON đã lưu sẵn.

Các phần chính:
- Dataset audit cho 5% và 10% labeled split, dựng trực tiếp từ dataset YOLO gốc bằng logic Notebook 01.
- Bảng metric tổng hợp với tên phương pháp nghiên cứu thay vì mã NB02–NB06.
- Bảng summary kiểu báo cáo: Method / Precision / Recall / F1 / mAP, tự in đậm giá trị tốt nhất.
- Plot so sánh Val/Test AP, AP50, AP75, APs.
- Learning curves, pseudo-label diagnostics, và focused diagnostics cho Sup-only vs Full model.
- Confusion matrix dạng YOLO-style: raw và normalized nằm cạnh nhau, màu nhẹ hơn.
- Biểu đồ cột per-class mAP/AP50/AP75 theo từng ratio và từng model.
- Qualitative prediction grids theo từng ratio, dùng prediction JSON đã lưu sẵn.


In [ ]:
from pathlib import Path
import json
import warnings
import random
import re
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

try:
    import yaml
except Exception:
    yaml = None

try:
    from sklearn.model_selection import StratifiedShuffleSplit
except Exception:
    StratifiedShuffleSplit = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

OUTPUT_DIR = Path("/kaggle/working/summary_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def compact_print(*args):
    print(*args)

def savefig(name, dpi=180):
    path = OUTPUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"saved: {name}")
    return path

print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 1. Auto-detect paths

In [ ]:
def find_summary_root():
    base = Path("/kaggle/input")
    candidates = []
    for p in base.rglob("*"):
        if p.is_dir() and (p / "ratio_05").exists() and (p / "ratio_10").exists():
            candidates.append(p)
    candidates = sorted(set(candidates), key=lambda x: (len(str(x)), str(x)))
    if not candidates:
        raise FileNotFoundError("Không tìm thấy folder có ratio_05 và ratio_10 trong /kaggle/input.")
    return candidates[0]

def find_yolo_root():
    # User-provided canonical path. Keep this first to avoid accidentally selecting summary inputs.
    preferred = [
        Path("/kaggle/input/datasets/minhquan228/my-pcb/YOLO_format"),
        Path("/kaggle/input/my_pcb/YOLO_format"),
        Path("/kaggle/input/my-pcb/YOLO_format"),
        Path("/kaggle/input/my_pcb/YOLO_format"),
    ]
    for p in preferred:
        if (p / "data.yaml").exists() and (p / "train").exists() and (p / "valid").exists() and (p / "test").exists():
            return p

    base = Path("/kaggle/input")
    candidates = []
    for data_yaml in base.rglob("data.yaml"):
        p = data_yaml.parent
        if (p / "train").exists() and (p / "valid").exists() and (p / "test").exists():
            candidates.append(p)
    candidates = sorted(set(candidates), key=lambda x: (len(str(x)), str(x)))
    return candidates[0] if candidates else None

SUMMARY_ROOT = find_summary_root()
YOLO_ROOT = find_yolo_root()

path_df = pd.DataFrame([
    {"Name": "SUMMARY_ROOT", "Path": str(SUMMARY_ROOT), "Exists": SUMMARY_ROOT.exists()},
    {"Name": "YOLO_ROOT", "Path": str(YOLO_ROOT), "Exists": YOLO_ROOT.exists() if YOLO_ROOT is not None else False},
])
display(path_df)

## 2. Run specification

In [ ]:
RUN_SPECS = [
    {
        "nb": "NB02",
        "method": "Supervised 10%/5% Deformable DETR",
        "display": "Def-DETR Sup-only",
        "folder": "nb02_supervised",
    },
    {
        "nb": "NB03A",
        "method": "Vanilla DETR-SSOD baseline",
        "display": "Def-DETR SSOD",
        "folder": "nb03a_vanilla_ssod",
    },
    {
        "nb": "NB04",
        "method": "DETR-SSOD + Stage-wise Hybrid Matching",
        "display": "DETR-SSOD + SHM",
        "folder": "nb04_shm",
    },
    {
        "nb": "NB05",
        "method": "Semi-DETR ablation: SHM + CQC",
        "display": "SHM + CQC",
        "folder": "nb05_shm_cqc",
    },
    {
        "nb": "NB06",
        "method": "Full Semi-DETR-style: SHM + CQC + CPM",
        "display": "Full SHM+CQC+CPM",
        "folder": "nb06_shm_cqc_cpm",
    },
]

RATIO_SPECS = [
    {"ratio": "5%", "ratio_value": 0.05, "folder": "ratio_05"},
    {"ratio": "10%", "ratio_value": 0.10, "folder": "ratio_10"},
]

runs = []
for ratio_spec in RATIO_SPECS:
    for idx, spec in enumerate(RUN_SPECS):
        run_dir = SUMMARY_ROOT / ratio_spec["folder"] / spec["folder"]
        item = dict(spec)
        item.update(ratio_spec)
        item["path"] = run_dir
        item["nb_order"] = idx
        runs.append(item)

pd.DataFrame([
    {
        "Ratio": r["ratio"],
        "Notebook": r["nb"],
        "Display name": r["display"],
        "Folder": str(r["path"]),
    }
    for r in runs
])

## 3. Validate files

In [ ]:
CORE_FILES = ["metrics.json", "config.json", "history.json"]

def pick_pred_file(run_dir: Path, split="test"):
    candidates = [
        f"pred_final_{split}.json",
        f"pred_final_{split}_student.json",
        f"pred_final_{split}_teacher.json",
    ]
    for name in candidates:
        path = run_dir / name
        if path.exists():
            return path, name
    return None, None

def validate_inputs(runs):
    rows = []
    for r in runs:
        run_dir = r["path"]
        missing_core = [f for f in CORE_FILES if not (run_dir / f).exists()]
        pred_test, pred_test_name = pick_pred_file(run_dir, "test")
        pred_val, pred_val_name = pick_pred_file(run_dir, "val")
        rows.append({
            "Ratio": r["ratio"],
            "Notebook": r["nb"],
            "Method": r["display"],
            "Run dir": str(run_dir),
            "Exists": run_dir.exists(),
            "Missing core": ", ".join(missing_core),
            "Test prediction": pred_test_name or "MISSING",
            "Val prediction": pred_val_name or "MISSING",
            "GT test": (run_dir / "gt_test_cache.json").exists(),
            "GT valid": (run_dir / "gt_valid_cache.json").exists(),
        })
    return pd.DataFrame(rows)

validation_df = validate_inputs(runs)
display(validation_df)

validation_df.to_csv(OUTPUT_DIR / "input_validation.csv", index=False)
if (validation_df["Missing core"].astype(str).str.len() > 0).any():
    print("warning: some runs are missing core files")
else:
    print("input validation: all core files found")

## 4. Dataset audit for 5% and 10% splits

Phần này trực quan hóa dữ liệu trực tiếp từ dataset YOLO gốc:

`/kaggle/input/datasets/minhquan228/my-pcb/YOLO_format`

Notebook sử dụng lại logic của Notebook 01:
- scan `train/valid/test` từ YOLO format;
- đọc label `.txt`;
- tạo stratified labeled/unlabeled split theo dominant class;
- dựng đồng thời split `5%` và `10%` bằng cùng seed;
- vẽ class distribution, imbalance, object/image, bbox area/aspect và sample ảnh.

Phần này **không cần split JSON** trong `semidetr_summary_inputs`.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SEED = 42
IMG_SIZE = 640
random.seed(SEED)
np.random.seed(SEED)

# Soft, report-friendly colors.
PALETTE = {
    "labeled": "#7fb3d5",
    "unlabeled": "#f5b7b1",
    "train": "#b3cde3",
    "valid": "#ccebc5",
    "test": "#decbe4",
}
CLASS_COLORS_AUDIT = ["#d62728", "#2ca02c", "#1f77b4", "#ffbf00", "#9467bd"]

def get_split_dirs(yolo_root, split):
    root = Path(yolo_root) / split
    img_dir = root / "images" if (root / "images").exists() else root
    lbl_dir = root / "labels" if (root / "labels").exists() else root
    return img_dir, lbl_dir

def load_class_names(yolo_root):
    yaml_file = Path(yolo_root) / "data.yaml"
    if yaml is not None and yaml_file.exists():
        with open(yaml_file, "r", encoding="utf-8") as f:
            data_cfg = yaml.safe_load(f)
        names = data_cfg.get("names", None)
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if isinstance(names, list) and len(names):
            return [str(x) for x in names]
        nc = int(data_cfg.get("nc", 5))
        return [str(i) for i in range(nc)]
    return [str(i) for i in range(5)]

if YOLO_ROOT is None or not Path(YOLO_ROOT).exists():
    raise FileNotFoundError("YOLO_ROOT không tồn tại. Kiểm tra lại /kaggle/input/datasets/minhquan228/my-pcb/YOLO_format")

CLASS_NAMES = load_class_names(YOLO_ROOT)
NUM_CLASSES = len(CLASS_NAMES)

def read_yolo_label(label_path):
    bboxes, classes = [], []
    label_path = Path(label_path)
    if not label_path.exists():
        return bboxes, classes
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return bboxes, classes
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            cx, cy, w, h = map(float, parts[1:5])
            classes.append(cls)
            bboxes.append((cx, cy, w, h))
        except Exception:
            continue
    return bboxes, classes

def scan_split(yolo_root, split):
    img_dir, lbl_dir = get_split_dirs(yolo_root, split)
    rows = []
    for img_path in sorted(img_dir.rglob("*")):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        bboxes, classes = read_yolo_label(lbl_path)
        rows.append({
            "split": split,
            "img_path": str(img_path),
            "lbl_path": str(lbl_path),
            "image_name": img_path.name,
            "stem": img_path.stem,
            "has_label": lbl_path.exists(),
            "n_obj": len(classes),
            "classes": classes,
            "bboxes": bboxes,
        })
    return pd.DataFrame(rows)

def dominant_class(classes):
    if not classes:
        return -1
    return Counter(classes).most_common(1)[0][0]

def create_stratified_split(df, ratio, seed=42):
    """Same logic as Notebook 01: stratify by dominant class; fallback to random."""
    df = df.copy()
    df["dom_class"] = df["classes"].apply(dominant_class)
    df_valid_for_split = df[df["n_obj"] > 0].copy()
    if len(df_valid_for_split) == 0:
        return df.iloc[:0].copy(), df.iloc[:0].copy(), "empty"
    try:
        if StratifiedShuffleSplit is None:
            raise RuntimeError("sklearn StratifiedShuffleSplit unavailable")
        sss = StratifiedShuffleSplit(n_splits=1, train_size=1.0 - ratio, random_state=seed)
        unlabeled_idx, labeled_idx = next(sss.split(df_valid_for_split, df_valid_for_split["dom_class"]))
        df_labeled = df_valid_for_split.iloc[labeled_idx].copy()
        df_unlabeled = df_valid_for_split.iloc[unlabeled_idx].copy()
        mode = "stratified"
    except Exception as e:
        n_labeled = max(1, int(round(len(df_valid_for_split) * ratio)))
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(df_valid_for_split))
        df_labeled = df_valid_for_split.iloc[idx[:n_labeled]].copy()
        df_unlabeled = df_valid_for_split.iloc[idx[n_labeled:]].copy()
        mode = f"random_fallback: {e}"
    return df_labeled, df_unlabeled, mode

def class_counter(df):
    cnt = Counter()
    for classes in df["classes"]:
        for c in classes:
            cnt[int(c)] += 1
    return cnt

def class_dist_rows(df, ratio, subset):
    cnt = class_counter(df)
    total = sum(cnt.values()) or 1
    rows = []
    for cls in range(NUM_CLASSES):
        n = int(cnt.get(cls, 0))
        rows.append({
            "Ratio": ratio,
            "Subset": subset,
            "Class": cls,
            "Class name": CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else str(cls),
            "Objects": n,
            "Percent": 100.0 * n / total,
            "Images": int(len(df)),
            "Total objects": int(sum(cnt.values())),
        })
    return rows

def bbox_long_rows(df, ratio, subset):
    rows = []
    for _, row in df.iterrows():
        for cls, (cx, cy, w, h) in zip(row["classes"], row["bboxes"]):
            rows.append({
                "Ratio": ratio,
                "Subset": subset,
                "Class": int(cls),
                "w_px": float(w * IMG_SIZE),
                "h_px": float(h * IMG_SIZE),
                "area_pct": float(w * h * 100.0),
                "aspect": float(w / h) if h > 0 else np.nan,
            })
    return rows

def object_count_rows(df, ratio, subset):
    return [{"Ratio": ratio, "Subset": subset, "Objects": int(x)} for x in df["n_obj"].tolist()]

def add_bar_labels(ax, fmt="{:.0f}", fontsize=8, rotation=0):
    for container in ax.containers:
        try:
            ax.bar_label(container, fmt=fmt, fontsize=fontsize, rotation=rotation, padding=2)
        except Exception:
            pass

# Scan dataset directly from YOLO root.
df_train = scan_split(YOLO_ROOT, "train")
df_valid = scan_split(YOLO_ROOT, "valid")
df_test = scan_split(YOLO_ROOT, "test")

inventory_df = pd.DataFrame([
    {"Split": "train", "Images": len(df_train), "Label files": int(df_train["has_label"].sum()), "Missing labels": int((~df_train["has_label"]).sum()), "Objects": int(df_train["n_obj"].sum())},
    {"Split": "valid", "Images": len(df_valid), "Label files": int(df_valid["has_label"].sum()), "Missing labels": int((~df_valid["has_label"]).sum()), "Objects": int(df_valid["n_obj"].sum())},
    {"Split": "test", "Images": len(df_test), "Label files": int(df_test["has_label"].sum()), "Missing labels": int((~df_test["has_label"]).sum()), "Objects": int(df_test["n_obj"].sum())},
])

display(pd.DataFrame([{
    "YOLO_ROOT": str(YOLO_ROOT),
    "Classes": NUM_CLASSES,
    "Class names": ", ".join(CLASS_NAMES),
    "Seed": SEED,
}]))
display(inventory_df)
inventory_df.to_csv(OUTPUT_DIR / "dataset_inventory.csv", index=False)

# Create both 5% and 10% splits using Notebook 01 logic.
split_map = {}
dataset_audit_rows, split_stats_rows, obj_count_rows, bbox_rows = [], [], [], []

for ratio_value, ratio_name in [(0.05, "5%"), (0.10, "10%")]:
    df_labeled, df_unlabeled, split_mode = create_stratified_split(df_train, ratio_value, SEED)
    split_map[ratio_name] = {"labeled": df_labeled, "unlabeled": df_unlabeled, "mode": split_mode}

    split_artifact = {
        "seed": SEED,
        "labeled_ratio": ratio_value,
        "split_mode": split_mode,
        "n_labeled": int(len(df_labeled)),
        "n_unlabeled": int(len(df_unlabeled)),
        "n_train": int(len(df_train)),
        "n_valid": int(len(df_valid)),
        "n_test": int(len(df_test)),
        "labeled_stems": sorted(df_labeled["stem"].tolist()),
        "unlabeled_stems": sorted(df_unlabeled["stem"].tolist()),
        "valid_stems": sorted(df_valid["stem"].tolist()),
        "test_stems": sorted(df_test["stem"].tolist()),
        "class_names": CLASS_NAMES,
        "nc": NUM_CLASSES,
        "img_size": IMG_SIZE,
        "source": "reconstructed_from_yolo_root_using_nb01_logic",
    }
    with open(OUTPUT_DIR / f"reconstructed_split_{ratio_name.replace('%','pct')}_seed{SEED}.json", "w", encoding="utf-8") as f:
        json.dump(split_artifact, f, indent=2, ensure_ascii=False)

    for subset, df_sub in [("labeled", df_labeled), ("unlabeled", df_unlabeled)]:
        dataset_audit_rows.extend(class_dist_rows(df_sub, ratio_name, subset))
        split_stats_rows.append({
            "Ratio": ratio_name,
            "Subset": subset,
            "Split mode": split_mode,
            "Images": int(len(df_sub)),
            "Image percent": float(len(df_sub) / max(len(df_train), 1) * 100.0),
            "Objects": int(df_sub["n_obj"].sum()),
            "Objects/image mean": float(df_sub["n_obj"].mean()) if len(df_sub) else 0.0,
            "Objects/image median": float(df_sub["n_obj"].median()) if len(df_sub) else 0.0,
        })
        obj_count_rows.extend(object_count_rows(df_sub, ratio_name, subset))
        bbox_rows.extend(bbox_long_rows(df_sub, ratio_name, subset))

context_rows = []
for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    context_rows.extend(class_dist_rows(df, "all", split_name))

context_class_df = pd.DataFrame(context_rows)
dataset_audit_df = pd.DataFrame(dataset_audit_rows)
split_stats_df = pd.DataFrame(split_stats_rows)
object_count_df = pd.DataFrame(obj_count_rows)
bbox_df = pd.DataFrame(bbox_rows)

balance_rows = []
for ratio in ["5%", "10%"]:
    for subset in ["labeled", "unlabeled"]:
        sub = dataset_audit_df[(dataset_audit_df["Ratio"] == ratio) & (dataset_audit_df["Subset"] == subset)]
        if len(sub):
            max_pct = float(sub["Percent"].max())
            min_pct = float(sub["Percent"].min())
            max_obj = int(sub["Objects"].max())
            min_obj = int(sub["Objects"].min())
            balance_rows.append({
                "Ratio": ratio,
                "Subset": subset,
                "Images": int(sub["Images"].iloc[0]),
                "Total objects": int(sub["Total objects"].iloc[0]),
                "Max objects": max_obj,
                "Min objects": min_obj,
                "Max/Min objects": max_obj / min_obj if min_obj > 0 else np.nan,
                "Max percent": max_pct,
                "Min percent": min_pct,
                "Max/Min percent": max_pct / min_pct if min_pct > 0 else np.nan,
            })
balance_df = pd.DataFrame(balance_rows)

display(split_stats_df)
display(balance_df)
display(dataset_audit_df)

context_class_df.to_csv(OUTPUT_DIR / "dataset_context_class_distribution.csv", index=False)
dataset_audit_df.to_csv(OUTPUT_DIR / "dataset_audit_class_distribution.csv", index=False)
split_stats_df.to_csv(OUTPUT_DIR / "dataset_split_stats.csv", index=False)
balance_df.to_csv(OUTPUT_DIR / "dataset_balance_summary.csv", index=False)
bbox_df.to_csv(OUTPUT_DIR / "dataset_bbox_stats_long.csv", index=False)
object_count_df.to_csv(OUTPUT_DIR / "dataset_object_count_long.csv", index=False)

# Plot A: inventory with exact counts.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
bars0 = axes[0].bar(inventory_df["Split"], inventory_df["Images"], color=[PALETTE["train"], PALETTE["valid"], PALETTE["test"]])
axes[0].set_title("Dataset images by split")
axes[0].set_ylabel("Images")
axes[0].grid(axis="y", alpha=0.25)
axes[0].bar_label(bars0, labels=[f"{int(x):,}" for x in inventory_df["Images"]], fontsize=9, padding=3)

bars1 = axes[1].bar(inventory_df["Split"], inventory_df["Objects"], color=[PALETTE["train"], PALETTE["valid"], PALETTE["test"]])
axes[1].set_title("Dataset objects by split")
axes[1].set_ylabel("Objects")
axes[1].grid(axis="y", alpha=0.25)
axes[1].bar_label(bars1, labels=[f"{int(x):,}" for x in inventory_df["Objects"]], fontsize=9, padding=3)
savefig("dataset_inventory_split_counts.png")
plt.show()

# Plot B: train/valid/test class distribution with object counts.
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.0))
plot_df = context_class_df[context_class_df["Ratio"] == "all"]
pct_piv = plot_df.pivot(index="Class", columns="Subset", values="Percent").reindex(range(NUM_CLASSES)).fillna(0)
cnt_piv = plot_df.pivot(index="Class", columns="Subset", values="Objects").reindex(range(NUM_CLASSES)).fillna(0)

pct_piv[[c for c in ["train", "valid", "test"] if c in pct_piv.columns]].plot(kind="bar", ax=axes[0], color=[PALETTE["train"], PALETTE["valid"], PALETTE["test"]])
axes[0].set_title("Class distribution across train / valid / test (%)")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Object percentage (%)")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(title="")
cnt_piv[[c for c in ["train", "valid", "test"] if c in cnt_piv.columns]].plot(kind="bar", ax=axes[1], color=[PALETTE["train"], PALETTE["valid"], PALETTE["test"]])
axes[1].set_title("Class object counts across train / valid / test")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Objects")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(title="")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
savefig("dataset_context_class_distribution_percent_and_counts.png")
plt.show()

# Plot C: labeled/unlabeled class distribution per ratio, with exact object counts.
fig, axes = plt.subplots(2, 2, figsize=(16, 9.2), sharex=True)
for col, ratio in enumerate(["5%", "10%"]):
    sub = dataset_audit_df[dataset_audit_df["Ratio"] == ratio]
    pct_piv = sub.pivot(index="Class", columns="Subset", values="Percent").reindex(range(NUM_CLASSES)).fillna(0)
    cnt_piv = sub.pivot(index="Class", columns="Subset", values="Objects").reindex(range(NUM_CLASSES)).fillna(0)

    pct_piv[[c for c in ["labeled", "unlabeled"] if c in pct_piv.columns]].plot(kind="bar", ax=axes[0, col], color=[PALETTE["labeled"], PALETTE["unlabeled"]])
    axes[0, col].set_title(f"Labeled vs unlabeled class distribution ({ratio})")
    axes[0, col].set_ylabel("Object percentage (%)")
    axes[0, col].grid(axis="y", alpha=0.25)
    axes[0, col].legend(title="")
    axes[0, col].tick_params(axis="x", rotation=0)

    cnt_piv[[c for c in ["labeled", "unlabeled"] if c in cnt_piv.columns]].plot(kind="bar", ax=axes[1, col], color=[PALETTE["labeled"], PALETTE["unlabeled"]])
    axes[1, col].set_title(f"Labeled vs unlabeled object counts ({ratio})")
    axes[1, col].set_xlabel("Class")
    axes[1, col].set_ylabel("Objects")
    axes[1, col].grid(axis="y", alpha=0.25)
    axes[1, col].legend(title="")
    axes[1, col].tick_params(axis="x", rotation=0)
savefig("dataset_labeled_unlabeled_class_distribution_percent_and_counts.png")
plt.show()

# Plot D: class gaps and imbalance ratio.
fig, axes = plt.subplots(2, 2, figsize=(16, 8.7))
for col, ratio in enumerate(["5%", "10%"]):
    sub = dataset_audit_df[dataset_audit_df["Ratio"] == ratio]
    pct_piv = sub.pivot(index="Class", columns="Subset", values="Percent").reindex(range(NUM_CLASSES)).fillna(0)
    cnt_piv = sub.pivot(index="Class", columns="Subset", values="Objects").reindex(range(NUM_CLASSES)).fillna(0)

    pct_diff = pct_piv.get("labeled", 0) - pct_piv.get("unlabeled", 0)
    cnt_diff = cnt_piv.get("labeled", 0) - cnt_piv.get("unlabeled", 0)

    axes[0, col].bar(pct_diff.index.astype(str), pct_diff.values, color="#a9cce3")
    axes[0, col].axhline(0, color="gray", linewidth=1)
    axes[0, col].set_title(f"Labeled − unlabeled percentage gap ({ratio})")
    axes[0, col].set_ylabel("Percentage-point difference")
    axes[0, col].grid(axis="y", alpha=0.25)

    axes[1, col].bar(cnt_diff.index.astype(str), cnt_diff.values, color="#d7bde2")
    axes[1, col].axhline(0, color="gray", linewidth=1)
    axes[1, col].set_title(f"Labeled − unlabeled object-count gap ({ratio})")
    axes[1, col].set_xlabel("Class")
    axes[1, col].set_ylabel("Object-count difference")
    axes[1, col].grid(axis="y", alpha=0.25)
savefig("dataset_labeled_unlabeled_class_gap_percent_and_count.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for subset, color in [("labeled", PALETTE["labeled"]), ("unlabeled", PALETTE["unlabeled"])]:
    sub = balance_df[balance_df["Subset"] == subset]
    axes[0].plot(sub["Ratio"], sub["Max/Min percent"], marker="o", linewidth=2, label=subset, color=color)
    axes[1].plot(sub["Ratio"], sub["Max/Min objects"], marker="o", linewidth=2, label=subset, color=color)
axes[0].axhline(1.0, color="gray", linewidth=1, linestyle="--")
axes[0].set_title("Class imbalance ratio by percentage")
axes[0].set_ylabel("Max class % / Min class %")
axes[0].grid(alpha=0.25)
axes[0].legend(title="")
axes[1].axhline(1.0, color="gray", linewidth=1, linestyle="--")
axes[1].set_title("Class imbalance ratio by object count")
axes[1].set_ylabel("Max class objects / Min class objects")
axes[1].grid(alpha=0.25)
axes[1].legend(title="")
savefig("dataset_imbalance_ratio_percent_and_count.png")
plt.show()

# Plot E: split size and object count.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
img_piv = split_stats_df.pivot(index="Ratio", columns="Subset", values="Images").reindex(["5%", "10%"])
obj_piv = split_stats_df.pivot(index="Ratio", columns="Subset", values="Objects").reindex(["5%", "10%"])
img_piv.plot(kind="bar", stacked=True, ax=axes[0], color=[PALETTE["labeled"], PALETTE["unlabeled"]])
axes[0].set_title("Image split size")
axes[0].set_xlabel("Ratio")
axes[0].set_ylabel("Images")
axes[0].grid(axis="y", alpha=0.25)
axes[0].tick_params(axis="x", rotation=0)
obj_piv.plot(kind="bar", stacked=True, ax=axes[1], color=[PALETTE["labeled"], PALETTE["unlabeled"]])
axes[1].set_title("Object count by split")
axes[1].set_xlabel("Ratio")
axes[1].set_ylabel("Objects")
axes[1].grid(axis="y", alpha=0.25)
axes[1].tick_params(axis="x", rotation=0)
for ax, piv in zip(axes, [img_piv, obj_piv]):
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", label_type="center", fontsize=8, color="black")
savefig("dataset_split_size_object_count.png")
plt.show()

# Plot F: object/image and bbox distributions.
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
for subset, color in [("labeled", PALETTE["labeled"]), ("unlabeled", PALETTE["unlabeled"])]:
    sub = object_count_df[object_count_df["Subset"] == subset]
    axes[0].hist(sub[sub["Ratio"] == "5%"]["Objects"], bins=30, alpha=0.65, label=f"5% {subset}", color=color)
axes[0].set_title("Objects per image — 5% split")
axes[0].set_xlabel("Objects/image")
axes[0].set_ylabel("Frequency")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)
for subset, color in [("labeled", PALETTE["labeled"]), ("unlabeled", PALETTE["unlabeled"])]:
    sub = object_count_df[object_count_df["Subset"] == subset]
    axes[1].hist(sub[sub["Ratio"] == "10%"]["Objects"], bins=30, alpha=0.65, label=f"10% {subset}", color=color)
axes[1].set_title("Objects per image — 10% split")
axes[1].set_xlabel("Objects/image")
axes[1].set_ylabel("Frequency")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
savefig("dataset_objects_per_image_distribution.png")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17.5, 4.8))
train_bbox = pd.DataFrame(bbox_long_rows(df_train, "all", "train"))
if len(train_bbox):
    axes[0].hist(train_bbox["w_px"], bins=50, color="#4c78a8", alpha=0.78, edgecolor="white", linewidth=0.4)
    axes[0].set_title("BBox width distribution")
    axes[0].set_xlabel("Width (px)")
    axes[0].grid(alpha=0.25)
    axes[1].hist(train_bbox["h_px"], bins=50, color="#59a14f", alpha=0.78, edgecolor="white", linewidth=0.4)
    axes[1].set_title("BBox height distribution")
    axes[1].set_xlabel("Height (px)")
    axes[1].grid(alpha=0.25)
    axes[2].hist(train_bbox["area_pct"], bins=50, color="#e15759", alpha=0.78, edgecolor="white", linewidth=0.4)
    axes[2].set_title("BBox area distribution")
    axes[2].set_xlabel("Area (% image)")
    axes[2].grid(alpha=0.25)
savefig("dataset_bbox_distribution_train.png")
plt.show()

fig, ax = plt.subplots(figsize=(9.5, 4.8))
if len(bbox_df):
    labels = []
    values = []
    for ratio in ["5%", "10%"]:
        for subset in ["labeled", "unlabeled"]:
            vals = bbox_df[(bbox_df["Ratio"] == ratio) & (bbox_df["Subset"] == subset)]["area_pct"].values
            if len(vals):
                labels.append(f"{ratio}\n{subset}")
                values.append(vals)
    ax.boxplot(values, labels=labels, showfliers=False, patch_artist=True)
    ax.set_title("BBox area distribution by ratio/subset")
    ax.set_ylabel("BBox area (% image)")
    ax.grid(axis="y", alpha=0.25)
savefig("dataset_bbox_area_boxplot_by_ratio.png")
plt.show()

# Plot G: compact sample images: 4 images per ratio, 2 labeled + 2 unlabeled.
def draw_yolo_sample(ax, row, title=""):
    img = Image.open(row["img_path"]).convert("RGB")
    W, H = img.size
    ax.imshow(img)
    ax.set_title(title, fontsize=9, pad=2)
    ax.axis("off")
    for cls, (cx, cy, w, h) in zip(row["classes"], row["bboxes"]):
        x1 = (cx - w / 2) * W
        y1 = (cy - h / 2) * H
        color = CLASS_COLORS_AUDIT[int(cls) % len(CLASS_COLORS_AUDIT)]
        ax.add_patch(Rectangle((x1, y1), w * W, h * H, linewidth=1.5, edgecolor=color, facecolor="none"))
        label = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else str(cls)
        ax.text(x1, max(0, y1 - 3), label, fontsize=6.5, color="black", bbox=dict(facecolor="white", edgecolor=color, alpha=0.72, pad=0.5))

for ratio in ["5%", "10%"]:
    selected = []
    for subset in ["labeled", "unlabeled"]:
        df_sub = split_map[ratio][subset]
        if len(df_sub):
            samp = df_sub.sample(min(2, len(df_sub)), random_state=SEED)
            for i, (_, row) in enumerate(samp.iterrows(), start=1):
                selected.append((subset, i, row))

    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), squeeze=False)
    for c, item in enumerate(selected[:4]):
        subset, idx, row = item
        draw_yolo_sample(axes[0, c], row, title=f"Sample - {subset} {idx}")
    for c in range(len(selected), 4):
        axes[0, c].axis("off")
    fig.suptitle(f"Dataset samples with GT boxes — {ratio}", fontsize=13, y=0.99)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.86, bottom=0.01, wspace=0.02, hspace=0.02)
    out = OUTPUT_DIR / f"dataset_samples_{ratio.replace('%','pct')}.png"
    plt.savefig(out, dpi=190, bbox_inches="tight", pad_inches=0.02)
    print(f"saved: {out.name}")
    plt.show()

## 4. Helper functions for loading metrics

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten_dict(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            nk = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                out.update(flatten_dict(v, nk))
            else:
                out[nk] = v
    return out

def norm_metric_name(k):
    s = str(k).lower().replace("_", "").replace("-", "").replace("@", "")
    mapping = {
        "precision": "Precision", "prec": "Precision", "p": "Precision",
        "recall": "Recall", "rec": "Recall", "r": "Recall",
        "f1": "F1", "f1score": "F1",
        "ap": "AP", "map": "AP", "map5095": "AP", "ap5095": "AP",
        "ap50": "AP50", "map50": "AP50",
        "ap75": "AP75", "map75": "AP75",
        "aps": "APs", "apsmall": "APs",
        "apm": "APm", "apmedium": "APm",
        "apl": "APl", "aplarge": "APl",
    }
    return mapping.get(s)

def extract_metric_dict(obj, split):
    split_terms = {
        "val": ["val", "valid", "validation", "best_val"],
        "test": ["test"]
    }[split]
    candidates = []

    def walk(x, path=""):
        if isinstance(x, dict):
            m = {}
            for k, v in x.items():
                nk = norm_metric_name(k)
                if nk and isinstance(v, (int, float)):
                    m[nk] = float(v)
            if m:
                pl = path.lower()
                score = sum(1 for t in split_terms if t in pl)
                if split == "test" and "test" in pl:
                    score += 10
                if split == "val" and ("val" in pl or "valid" in pl):
                    score += 10
                candidates.append((score, path, m))
            for k, v in x.items():
                walk(v, f"{path}.{k}" if path else str(k))
        elif isinstance(x, list):
            for i, v in enumerate(x):
                walk(v, f"{path}[{i}]")
    walk(obj)
    candidates = sorted(candidates, key=lambda z: (-z[0], z[1]))
    if candidates:
        return candidates[0][2], candidates[0][1]
    return {}, "not_found"

def history_to_df(x):
    if isinstance(x, list):
        return pd.DataFrame(x)
    if isinstance(x, dict):
        for key in ["history", "records", "logs", "epochs"]:
            if isinstance(x.get(key), list):
                return pd.DataFrame(x[key])
        if len(x) and all(isinstance(v, list) for v in x.values()):
            return pd.DataFrame(x)
    return pd.DataFrame()

def get_best_epoch(metrics, history_df):
    flat = flatten_dict(metrics)
    for k, v in flat.items():
        if "best" in k.lower() and "epoch" in k.lower() and isinstance(v, (int, float)):
            return int(v)
    # fallback from history
    if history_df is not None and len(history_df):
        cols = {str(c).lower(): c for c in history_df.columns}
        ep_col = cols.get("epoch", cols.get("ep"))
        ap_col = None
        for c in history_df.columns:
            cl = str(c).lower()
            if cl in ["val_ap", "primary_ap", "prim", "ap"] or "val_ap" in cl:
                ap_col = c
                break
        if ep_col is not None and ap_col is not None:
            tmp = history_df[[ep_col, ap_col]].copy()
            tmp[ap_col] = pd.to_numeric(tmp[ap_col], errors="coerce")
            tmp = tmp.dropna()
            if len(tmp):
                return int(tmp.loc[tmp[ap_col].idxmax(), ep_col])
    return None

## 5. Build summary dataframe

In [ ]:
summary_rows = []
histories = {}

for r in runs:
    run_dir = r["path"]
    if not all((run_dir / f).exists() for f in CORE_FILES):
        print("Skip missing core:", r["ratio"], r["nb"], run_dir)
        continue

    metrics = load_json(run_dir / "metrics.json")
    config = load_json(run_dir / "config.json")
    history_json = load_json(run_dir / "history.json")
    hist_df = history_to_df(history_json)
    histories[(r["ratio"], r["nb"])] = hist_df

    val_m, val_source = extract_metric_dict(metrics, "val")
    test_m, test_source = extract_metric_dict(metrics, "test")
    pred_test, pred_test_name = pick_pred_file(run_dir, "test")
    pred_val, pred_val_name = pick_pred_file(run_dir, "val")

    row = {
        "Ratio": r["ratio"],
        "Ratio value": r["ratio_value"],
        "Notebook": r["nb"],
        "Method": r["method"],
        "Display": r["display"],
        "Run dir": str(run_dir),
        "Best epoch": get_best_epoch(metrics, hist_df),
        "Val source": val_source,
        "Test source": test_source,
        "Pred test source": pred_test_name,
        "Pred val source": pred_val_name,
    }
    for k in ["Precision", "Recall", "F1", "AP", "AP50", "AP75", "APs", "APm", "APl"]:
        row[f"Val {k}"] = val_m.get(k, np.nan)
        row[f"Test {k}"] = test_m.get(k, np.nan)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
nb_order = {x["nb"]: i for i, x in enumerate(RUN_SPECS)}
ratio_order = {"5%": 0, "10%": 1}
summary_df["nb_order"] = summary_df["Notebook"].map(nb_order)
summary_df["ratio_order"] = summary_df["Ratio"].map(ratio_order)
summary_df = summary_df.sort_values(["ratio_order", "nb_order"]).reset_index(drop=True)

for metric in ["Val AP", "Test AP", "Test AP50", "Test AP75", "Test APs"]:
    col = f"Δ {metric} vs NB02"
    summary_df[col] = np.nan
    for ratio in summary_df["Ratio"].unique():
        mask = summary_df["Ratio"] == ratio
        base = summary_df.loc[mask & (summary_df["Notebook"] == "NB02"), metric]
        if len(base) and pd.notna(base.iloc[0]):
            summary_df.loc[mask, col] = summary_df.loc[mask, metric] - float(base.iloc[0])

main_cols = [
    "Ratio", "Display", "Method", "Best epoch",
    "Val AP", "Val AP50", "Val AP75", "Val APs",
    "Test Precision", "Test Recall", "Test F1",
    "Test AP", "Test AP50", "Test AP75", "Test APs",
    "Δ Test AP vs NB02", "Pred test source"
]
display(summary_df[[c for c in main_cols if c in summary_df.columns]])

summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
summary_df.to_json(OUTPUT_DIR / "summary_metrics.json", orient="records", indent=2, force_ascii=False)
print("Saved summary_metrics.csv/json")

## 6. Summary plots

In [ ]:
def savefig(name, dpi=180):
    path = OUTPUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"saved: {name}")
    return path

DISPLAY_MAP = {x["nb"]: x["display"] for x in RUN_SPECS}

def display_ordered_df(metric):
    sub = summary_df.sort_values(["ratio_order", "nb_order"]).copy()
    sub["DisplayLabel"] = sub["Notebook"].map(DISPLAY_MAP)
    return sub

def plot_progress(metric, title, filename):
    plt.figure(figsize=(11, 5.2))
    for ratio in ["5%", "10%"]:
        sub = summary_df[summary_df["Ratio"] == ratio].sort_values("nb_order").copy()
        sub["DisplayLabel"] = sub["Notebook"].map(DISPLAY_MAP)
        plt.plot(sub["DisplayLabel"], sub[metric], marker="o", linewidth=2, label=ratio)
        for x, y in zip(sub["DisplayLabel"], sub[metric]):
            if pd.notna(y):
                plt.text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=9)
    plt.title(title)
    plt.ylabel(metric)
    plt.grid(alpha=0.22)
    plt.legend(title="Ratio")
    plt.xticks(rotation=0)
    savefig(filename)
    plt.show()

def plot_grouped_bar(metric, title, filename):
    tmp = summary_df.copy()
    tmp["DisplayLabel"] = tmp["Notebook"].map(DISPLAY_MAP)
    order = [x["display"] for x in RUN_SPECS]
    piv = tmp.pivot(index="DisplayLabel", columns="Ratio", values=metric).reindex(order)
    ax = piv.plot(kind="bar", figsize=(11, 5.2), width=0.78)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.22)
    plt.xticks(rotation=0)
    savefig(filename)
    plt.show()

plot_progress("Test AP", "Test AP progression: supervised → full Semi-DETR-style", "summary_test_ap_progression.png")
plot_grouped_bar("Test AP", "Test AP comparison: 5% vs 10%", "summary_test_ap_bar.png")
plot_progress("Δ Test AP vs NB02", "Delta Test AP vs supervised baseline", "summary_delta_test_ap_vs_nb02.png")
plot_progress("Val AP", "Validation AP progression: supervised → full Semi-DETR-style", "summary_val_ap_progression.png")

for metric in ["Test AP50", "Test AP75", "Test APs"]:
    plot_progress(metric, f"{metric} progression", f"summary_{metric.lower().replace(' ', '_')}_progression.png")

## 6.1. Val/Test metric comparison by method

Phần này bổ sung bảng và biểu đồ so sánh **Validation** và **Test** cho từng phương pháp.  
Mục tiêu là nhìn rõ model nào tăng ở validation nhưng không tăng ở test, và metric nào cải thiện thật sự trên test set.


In [ ]:

# Core metrics for direct Val/Test comparison.
VAL_TEST_METRICS = [
    ("Precision", "Precision"),
    ("Recall", "Recall"),
    ("F1", "F1"),
    ("AP", "mAP50-95 / AP"),
    ("AP50", "mAP50"),
    ("AP75", "mAP75"),
    ("APs", "AP-small"),
]

# Keep only metrics that exist in the current summary dataframe.
available_val_test_metrics = []
for key, label in VAL_TEST_METRICS:
    val_col = f"Val {key}"
    test_col = f"Test {key}"
    if val_col in summary_df.columns or test_col in summary_df.columns:
        if summary_df[[c for c in [val_col, test_col] if c in summary_df.columns]].notna().any().any():
            available_val_test_metrics.append((key, label))

# Build a report-style table with both validation and test metrics.
val_test_table = summary_df.sort_values(["ratio_order", "nb_order"]).copy()
val_test_table["Model"] = val_test_table["Display"]

val_test_cols = ["Ratio", "Model"]
for key, label in available_val_test_metrics:
    for split in ["Val", "Test"]:
        col = f"{split} {key}"
        if col in val_test_table.columns:
            pretty = f"{split} {label}"
            val_test_table[pretty] = val_test_table[col]
            val_test_cols.append(pretty)

val_test_table = val_test_table[val_test_cols]
val_test_table.to_csv(OUTPUT_DIR / "val_test_metric_comparison.csv", index=False)

# Display table with best value per ratio and metric highlighted.
def bold_best_by_ratio(styler, df, numeric_cols):
    def highlight_group(group):
        styles = pd.DataFrame("", index=group.index, columns=group.columns)
        for col in numeric_cols:
            vals = pd.to_numeric(group[col], errors="coerce")
            if vals.notna().any():
                best = vals.max()
                styles.loc[vals == best, col] = "font-weight: bold"
        return styles
    return styler.apply(lambda _: pd.concat([highlight_group(g) for _, g in df.groupby("Ratio")]).sort_index(), axis=None)

numeric_cols = [c for c in val_test_table.columns if c not in ["Ratio", "Model"]]
try:
    styled_val_test = val_test_table.style.format({c: "{:.4f}" for c in numeric_cols})
    styled_val_test = bold_best_by_ratio(styled_val_test, val_test_table, numeric_cols)
    display(styled_val_test)
except Exception:
    display(val_test_table)

print(f"saved: val_test_metric_comparison.csv")

# Plot Val vs Test for each metric and each ratio.
def plot_val_test_bars_for_ratio(ratio, metric_key, metric_label):
    val_col = f"Val {metric_key}"
    test_col = f"Test {metric_key}"
    cols = [c for c in [val_col, test_col] if c in summary_df.columns]
    if len(cols) == 0:
        return

    sub = summary_df[summary_df["Ratio"] == ratio].sort_values("nb_order").copy()
    if sub.empty:
        return
    if not sub[cols].notna().any().any():
        return

    labels = sub["Display"].tolist()
    x = np.arange(len(labels))
    width = 0.36

    plt.figure(figsize=(13.5, 5.4))
    if val_col in sub.columns:
        plt.bar(x - width/2, sub[val_col], width, label="Validation")
        for xi, yi in zip(x - width/2, sub[val_col]):
            if pd.notna(yi):
                plt.text(xi, yi, f"{yi:.3f}", ha="center", va="bottom", fontsize=8)
    if test_col in sub.columns:
        plt.bar(x + width/2, sub[test_col], width, label="Test")
        for xi, yi in zip(x + width/2, sub[test_col]):
            if pd.notna(yi):
                plt.text(xi, yi, f"{yi:.3f}", ha="center", va="bottom", fontsize=8)

    plt.title(f"{ratio} — {metric_label}: Validation vs Test")
    plt.ylabel(metric_label)
    plt.xticks(x, labels, rotation=0)
    plt.ylim(bottom=0, top=min(1.05, max(1.0, np.nanmax(sub[cols].to_numpy(dtype=float)) + 0.08)))
    plt.grid(axis="y", alpha=0.22)
    plt.legend()
    safe_metric = metric_key.lower().replace(" ", "_")
    safe_ratio = ratio.replace("%", "pct")
    savefig(f"val_test_compare_{safe_ratio}_{safe_metric}.png")
    plt.show()

for ratio in ["5%", "10%"]:
    for metric_key, metric_label in available_val_test_metrics:
        plot_val_test_bars_for_ratio(ratio, metric_key, metric_label)

# Compact grid for the most important COCO-style metrics.
def plot_val_test_core_metric_grid(ratio, metric_keys=("AP", "AP50", "AP75", "APs")):
    sub = summary_df[summary_df["Ratio"] == ratio].sort_values("nb_order").copy()
    if sub.empty:
        return
    metric_keys = [m for m in metric_keys if f"Val {m}" in sub.columns and f"Test {m}" in sub.columns]
    if not metric_keys:
        return

    fig, axes = plt.subplots(1, len(metric_keys), figsize=(5.0 * len(metric_keys), 5.2), sharey=True)
    if len(metric_keys) == 1:
        axes = [axes]
    labels = sub["Display"].tolist()
    x = np.arange(len(labels))
    width = 0.36

    for ax, m in zip(axes, metric_keys):
        val_col, test_col = f"Val {m}", f"Test {m}"
        ax.bar(x - width/2, sub[val_col], width, label="Val")
        ax.bar(x + width/2, sub[test_col], width, label="Test")
        ax.set_title(m)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right")
        ax.grid(axis="y", alpha=0.22)
        ax.set_ylim(0, 1.05)
        for xi, yi in zip(x - width/2, sub[val_col]):
            if pd.notna(yi):
                ax.text(xi, yi, f"{yi:.3f}", ha="center", va="bottom", fontsize=7)
        for xi, yi in zip(x + width/2, sub[test_col]):
            if pd.notna(yi):
                ax.text(xi, yi, f"{yi:.3f}", ha="center", va="bottom", fontsize=7)
    axes[0].set_ylabel("Score")
    axes[-1].legend(loc="lower right")
    fig.suptitle(f"{ratio} — Core metrics: Validation vs Test", y=1.02)
    safe_ratio = ratio.replace("%", "pct")
    savefig(f"val_test_core_metrics_grid_{safe_ratio}.png")
    plt.show()

for ratio in ["5%", "10%"]:
    plot_val_test_core_metric_grid(ratio)


## 7. Learning curves

In [ ]:
def find_col(df, candidates):
    if df is None or len(df.columns) == 0:
        return None
    lower = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c
    return None

def plot_history_metric(candidates, title, ylabel, filename_prefix):
    for ratio in ["5%", "10%"]:
        plt.figure(figsize=(12.5, 5.3))
        ok = False
        for spec in RUN_SPECS:
            df = histories.get((ratio, spec["nb"]), pd.DataFrame())
            ep_col = find_col(df, ["epoch", "ep"])
            m_col = find_col(df, candidates)
            if ep_col is None or m_col is None:
                continue
            x = pd.to_numeric(df[ep_col], errors="coerce")
            y = pd.to_numeric(df[m_col], errors="coerce")
            mask = x.notna() & y.notna()
            if mask.sum() == 0:
                continue
            plt.plot(x[mask], y[mask], label=spec["display"], linewidth=1.9)
            ok = True
        plt.title(f"{title} — {ratio}")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.grid(alpha=0.28)
        plt.legend(fontsize=8)
        if ok:
            savefig(f"{filename_prefix}_{ratio.replace('%','pct')}.png")
            plt.show()
        else:
            plt.close()
            print("No data:", title, ratio)

plot_history_metric(["val_ap", "primary_ap", "prim", "ap"], "Validation AP curve", "AP", "history_val_ap")
plot_history_metric(["loss", "total_loss"], "Training loss curve", "Loss", "history_loss")
plot_history_metric(["sup", "sup_loss", "supervised_loss"], "Supervised loss curve", "Loss", "history_sup_loss")
plot_history_metric(["unsup", "unsup_loss", "unsupervised_loss"], "Unsupervised loss curve", "Loss", "history_unsup_loss")
plot_history_metric(["pseudo", "pseudo/img", "pseudo_per_image"], "Pseudo labels per image", "Pseudo/image", "history_pseudo")

# Focused diagnostics: compare only Sup-only vs Full model, then show internal curves for Full model.
FOCUS_NBS = ["NB02", "NB06"]

def plot_focused_nb02_nb06():
    metric_sets = [
        (["val_ap", "primary_ap", "prim", "ap"], "Validation AP", "AP", "focused_nb02_nb06_val_ap"),
        (["loss", "total_loss"], "Training loss", "Loss", "focused_nb02_nb06_loss"),
    ]

    for ratio in ["5%", "10%"]:
        for candidates, title, ylabel, fname in metric_sets:
            plt.figure(figsize=(10.5, 4.8))
            ok = False
            for nb in FOCUS_NBS:
                spec = next(x for x in RUN_SPECS if x["nb"] == nb)
                df = histories.get((ratio, nb), pd.DataFrame())
                ep_col = find_col(df, ["epoch", "ep"])
                m_col = find_col(df, candidates)
                if ep_col is None or m_col is None:
                    continue
                x = pd.to_numeric(df[ep_col], errors="coerce")
                y = pd.to_numeric(df[m_col], errors="coerce")
                mask = x.notna() & y.notna()
                if mask.sum() == 0:
                    continue
                plt.plot(x[mask], y[mask], marker="o", markersize=2.5, linewidth=1.9, label=spec["display"])
                ok = True
            if ok:
                plt.title(f"{title}: Sup-only vs Full model — {ratio}")
                plt.xlabel("Epoch")
                plt.ylabel(ylabel)
                plt.grid(alpha=0.28)
                plt.legend()
                savefig(f"{fname}_{ratio.replace('%','pct')}.png")
                plt.show()
            else:
                plt.close()

def plot_full_model_internal_diagnostics():
    # These plots focus on NB06 only. They are useful for explaining how pseudo-labeling/CPM behaves.
    full_candidates = [
        (["sup", "sup_loss", "supervised_loss"], "Supervised loss"),
        (["unsup", "unsup_loss", "unsupervised_loss"], "Unsupervised loss"),
        (["cqc", "cqc_loss"], "CQC loss"),
        (["pseudo", "pseudo/img", "pseudo_per_image"], "Pseudo labels / image"),
        (["cpm_keep", "cpm_kept", "kept"], "CPM kept pseudo-labels"),
        (["thr", "threshold", "pseudo_threshold"], "Pseudo threshold"),
        (["w_u", "unsup_weight"], "Unsupervised weight"),
    ]

    for ratio in ["5%", "10%"]:
        df = histories.get((ratio, "NB06"), pd.DataFrame())
        if df is None or len(df) == 0:
            continue
        ep_col = find_col(df, ["epoch", "ep"])
        if ep_col is None:
            continue
        x = pd.to_numeric(df[ep_col], errors="coerce")

        plotted = 0
        fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0))
        axes = axes.flatten()
        for candidates, label in full_candidates:
            if plotted >= len(axes):
                break
            col = find_col(df, candidates)
            if col is None:
                continue
            y = pd.to_numeric(df[col], errors="coerce")
            mask = x.notna() & y.notna()
            if mask.sum() == 0:
                continue
            ax = axes[plotted]
            ax.plot(x[mask], y[mask], linewidth=1.9)
            ax.set_title(label)
            ax.set_xlabel("Epoch")
            ax.grid(alpha=0.28)
            plotted += 1

        for j in range(plotted, len(axes)):
            axes[j].axis("off")

        if plotted:
            fig.suptitle(f"Focused Full model diagnostics — {ratio}", fontsize=13, y=0.995)
            savefig(f"focused_full_model_internal_diagnostics_{ratio.replace('%','pct')}.png")
            plt.show()
        else:
            plt.close()

plot_focused_nb02_nb06()
plot_full_model_internal_diagnostics()

## 8. Optional detection diagnostics: confusion matrix and report-style metric table

Confusion matrix được vẽ theo kiểu report-friendly:
- Cột trái: raw confusion matrix.
- Cột phải: normalized confusion matrix.
- Tên model dùng tên nghiên cứu thay vì NB02–NB06.

Phần này cũng tạo bảng summary kiểu báo cáo với `Precision`, `Recall`, `F1`, `mAP50`, `mAP50-95`, `AP75`, `APs`. Các giá trị tốt nhất trong từng cột được in đậm.


In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def bbox_to_xyxy(b):
    x, y, w, h = map(float, b)
    return [x, y, x + w, y + h]

def normalize_pred(data):
    if isinstance(data, dict):
        for k in ["predictions", "annotations", "results", "detections"]:
            if isinstance(data.get(k), list):
                data = data[k]
                break
    out = []
    if not isinstance(data, list):
        return out
    for p in data:
        if not isinstance(p, dict):
            continue
        image_id = p.get("image_id", p.get("file_name", p.get("filename")))
        label = p.get("category_id", p.get("label", p.get("class_id")))
        score = p.get("score", p.get("confidence", p.get("conf", 1.0)))
        bbox = p.get("bbox", p.get("box"))
        if image_id is None or label is None or bbox is None:
            continue
        out.append({"image_id": str(image_id), "category_id": int(label), "score": float(score), "bbox": bbox_to_xyxy(bbox)})
    return out

def normalize_gt(data):
    anns = []
    if isinstance(data, dict):
        if isinstance(data.get("annotations"), list):
            anns = data["annotations"]
        else:
            for img, vals in data.items():
                if isinstance(vals, list):
                    for a in vals:
                        if isinstance(a, dict):
                            b = dict(a)
                            b.setdefault("image_id", img)
                            anns.append(b)
    elif isinstance(data, list):
        anns = data
    out = []
    for a in anns:
        if not isinstance(a, dict):
            continue
        image_id = a.get("image_id", a.get("file_name", a.get("filename")))
        label = a.get("category_id", a.get("label", a.get("class_id")))
        bbox = a.get("bbox", a.get("box"))
        if image_id is None or label is None or bbox is None:
            continue
        out.append({"image_id": str(image_id), "category_id": int(label), "bbox": bbox_to_xyxy(bbox)})
    return out

def extract_image_id_to_file(gt_data):
    mapping = {}
    if isinstance(gt_data, dict) and isinstance(gt_data.get("images"), list):
        for im in gt_data["images"]:
            if isinstance(im, dict):
                iid = im.get("id", im.get("image_id", im.get("file_name", im.get("filename"))))
                fn = im.get("file_name", im.get("filename", im.get("path")))
                if iid is not None and fn is not None:
                    mapping[str(iid)] = str(fn)
    return mapping

def match(preds, gts, iou_thr=0.5, conf_thr=0.25):
    preds = [p for p in preds if p["score"] >= conf_thr]
    byp, byg = defaultdict(list), defaultdict(list)
    for p in preds:
        byp[p["image_id"]].append(p)
    for g in gts:
        byg[g["image_id"]].append(g)

    records = []
    for img in sorted(set(byp) | set(byg)):
        used = set()
        pred_list = sorted(byp[img], key=lambda z: -z["score"])
        gt_list = byg[img]
        for p in pred_list:
            best_iou, best_j = 0, None
            for j, g in enumerate(gt_list):
                if j in used:
                    continue
                iou = box_iou(p["bbox"], g["bbox"])
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j is not None and best_iou >= iou_thr:
                used.add(best_j)
                records.append(("match", gt_list[best_j]["category_id"], p["category_id"], p["score"]))
            else:
                records.append(("fp", -1, p["category_id"], p["score"]))
        for j, g in enumerate(gt_list):
            if j not in used:
                records.append(("fn", g["category_id"], -1, 0.0))
    return records

def load_pred_gt(ratio, nb, split="test"):
    row = summary_df[(summary_df["Ratio"] == ratio) & (summary_df["Notebook"] == nb)]
    if len(row) == 0:
        return None, None, None, None
    run_dir = Path(row.iloc[0]["Run dir"])
    pred_path, pred_name = pick_pred_file(run_dir, split)
    gt_name = "gt_test_cache.json" if split == "test" else "gt_valid_cache.json"
    gt_path = run_dir / gt_name
    if pred_path is None or not gt_path.exists():
        return None, None, pred_name, None
    gt_data = load_json(gt_path)
    return normalize_pred(load_json(pred_path)), normalize_gt(gt_data), pred_name, gt_data

def confusion_matrix_from_preds(preds, gts, num_classes=5, conf_thr=0.25):
    recs = match(preds, gts, conf_thr=conf_thr)
    bg = num_classes
    mat = np.zeros((num_classes + 1, num_classes + 1), dtype=float)
    for typ, gt, pr, score in recs:
        gi = gt if gt >= 0 else bg
        pi = pr if pr >= 0 else bg
        if gi <= bg and pi <= bg:
            mat[gi, pi] += 1
    return mat

def normalize_confusion(mat):
    row_sum = mat.sum(axis=1, keepdims=True)
    return np.divide(mat, row_sum, out=np.zeros_like(mat, dtype=float), where=row_sum > 0)

def prf_from_confusion(mat, num_classes=5):
    # Class-wise TP/FP/FN, excluding background row/column from averaging.
    tp = np.diag(mat[:num_classes, :num_classes])
    fp = mat[num_classes, :num_classes] + (mat[:num_classes, :num_classes].sum(axis=0) - tp)
    fn = mat[:num_classes, num_classes] + (mat[:num_classes, :num_classes].sum(axis=1) - tp)
    precision_cls = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    recall_cls = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
    precision = float(np.nanmean(precision_cls)) if len(precision_cls) else np.nan
    recall = float(np.nanmean(recall_cls)) if len(recall_cls) else np.nan
    f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def annotate_confusion(ax, mat, normalized=False):
    vmax = mat.max() if mat.size and mat.max() > 0 else 1.0
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            text = f"{val:.2f}" if normalized and val > 0 else (f"{int(val)}" if (not normalized and val > 0) else "")
            if text:
                color = "white" if val > 0.55 * vmax else "black"
                ax.text(j, i, text, ha="center", va="center", fontsize=8, color=color)

def plot_confusion_pair(ratio, nb, num_classes=5, conf_thr=0.25):
    preds, gts, pred_name, _ = load_pred_gt(ratio, nb, "test")
    if preds is None or gts is None:
        print(f"skip confusion: {ratio} {nb}")
        return

    mat = confusion_matrix_from_preds(preds, gts, num_classes=num_classes, conf_thr=conf_thr)
    mat_norm = normalize_confusion(mat)
    labels = [str(i) for i in range(num_classes)] + ["BG"]
    method = DISPLAY_MAP.get(nb, nb).replace("\\n", " ")

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
    for ax, arr, title, normalized in [
        (axes[0], mat, "Confusion matrix", False),
        (axes[1], mat_norm, "Normalized confusion matrix", True),
    ]:
        im = ax.imshow(arr, aspect="auto", cmap="Blues", vmin=0)
        ax.set_title(title)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Ground truth")
        ax.set_xticks(range(num_classes + 1))
        ax.set_yticks(range(num_classes + 1))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        annotate_confusion(ax, arr, normalized=normalized)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"{ratio} — {method} — {pred_name} — conf ≥ {conf_thr}", fontsize=13)
    safe_method = re.sub(r"[^a-zA-Z0-9]+", "_", method).strip("_").lower()
    fname = f"confusion_pair_{ratio.replace('%','pct')}_{safe_method}.png"
    savefig(fname, dpi=180)
    plt.show()

# Report-friendly confusion matrices
confusion_targets = [("10%", "NB02"), ("10%", "NB06"), ("5%", "NB02")]
try:
    best5 = summary_df[summary_df["Ratio"] == "5%"].sort_values("Test AP", ascending=False).iloc[0]["Notebook"]
    if best5 != "NB02":
        confusion_targets.append(("5%", best5))
except Exception as e:
    print(f"skip best 5% confusion: {e}")

seen = set()
for ratio, nb in confusion_targets:
    key = (ratio, nb)
    if key in seen:
        continue
    seen.add(key)
    plot_confusion_pair(ratio, nb, conf_thr=0.25)


# Report-style summary table with P/R/F1 computed from final prediction JSON.
def build_detection_report_table(conf_thr=0.25, split="test"):
    rows = []
    for ratio in ["5%", "10%"]:
        for spec in RUN_SPECS:
            nb = spec["nb"]
            row = summary_df[(summary_df["Ratio"] == ratio) & (summary_df["Notebook"] == nb)]
            if len(row) == 0:
                continue

            preds, gts, pred_name, _ = load_pred_gt(ratio, nb, split)
            if preds is not None and gts is not None:
                mat = confusion_matrix_from_preds(preds, gts, num_classes=NUM_CLASSES, conf_thr=conf_thr)
                precision, recall, f1 = prf_from_confusion(mat, num_classes=NUM_CLASSES)
            else:
                precision = row.iloc[0].get(f"{split.capitalize()} Precision", np.nan)
                recall = row.iloc[0].get(f"{split.capitalize()} Recall", np.nan)
                f1 = row.iloc[0].get(f"{split.capitalize()} F1", np.nan)

            rows.append({
                "Ratio": ratio,
                "Model": spec["display"],
                "Prediction source": pred_name,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "mAP50-95": float(row.iloc[0].get(f"{split.capitalize()} AP", np.nan)),
                "mAP50": float(row.iloc[0].get(f"{split.capitalize()} AP50", np.nan)),
                "mAP75": float(row.iloc[0].get(f"{split.capitalize()} AP75", np.nan)),
                "APs": float(row.iloc[0].get(f"{split.capitalize()} APs", np.nan)),
            })
    return pd.DataFrame(rows)

def bold_best_per_ratio(styler, df, numeric_cols):
    # apply per ratio so 5% and 10% are highlighted independently
    def highlight(col):
        styles = [""] * len(col)
        if col.name not in numeric_cols:
            return styles
        for ratio in df["Ratio"].unique():
            idx = df.index[df["Ratio"] == ratio]
            vals = pd.to_numeric(df.loc[idx, col.name], errors="coerce")
            if vals.notna().any():
                best = vals.max()
                for i in idx:
                    if pd.notna(df.loc[i, col.name]) and abs(float(df.loc[i, col.name]) - float(best)) < 1e-12:
                        styles[list(df.index).index(i)] = "font-weight: bold"
        return styles
    return styler.apply(highlight, axis=0)

report_table = build_detection_report_table(conf_thr=0.25, split="test")
report_table.to_csv(OUTPUT_DIR / "report_style_detection_summary_test.csv", index=False)

display_cols = ["Ratio", "Model", "Precision", "Recall", "F1", "mAP50-95", "mAP50", "mAP75", "APs", "Prediction source"]
numeric_cols = ["Precision", "Recall", "F1", "mAP50-95", "mAP50", "mAP75", "APs"]
styled_report = (
    report_table[display_cols]
    .style
    .format({c: "{:.4f}" for c in numeric_cols})
    .pipe(bold_best_per_ratio, report_table[display_cols], numeric_cols)
)
display(styled_report)

## 8.1. Per-class mAP bar charts

Phần này bỏ heatmap. Thay vào đó, notebook tính và vẽ **biểu đồ cột per-class mAP** cho từng class `0–4`.

Các chỉ số được tính từ `pred_final_test*.json` và `gt_test_cache.json`, không cần load `.pt`:
- `AP`: trung bình AP từ IoU `0.50:0.95`
- `AP50`: AP tại IoU `0.50`
- `AP75`: AP tại IoU `0.75`

Mục tiêu: xem với cùng một class, các model cải thiện localization/detection như thế nào.


In [ ]:

PER_CLASS_MAP_SPLIT = "test"
IOU_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)


def ap_from_precision_recall(precision, recall):
    """COCO-style 101-point interpolated AP for one class and one IoU threshold."""
    if len(precision) == 0 or len(recall) == 0:
        return np.nan
    precision = np.asarray(precision, dtype=float)
    recall = np.asarray(recall, dtype=float)
    # Monotonic precision envelope.
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    recall_points = np.linspace(0.0, 1.0, 101)
    ap_vals = []
    for r in recall_points:
        valid = precision[recall >= r]
        ap_vals.append(valid.max() if valid.size else 0.0)
    return float(np.mean(ap_vals))


def compute_class_ap(preds, gts, class_id, iou_thr=0.50):
    """Compute AP for one class at one IoU threshold from normalized prediction/GT lists."""
    cls_preds = [p for p in preds if int(p.get("category_id", -1)) == int(class_id)]
    cls_gts = [g for g in gts if int(g.get("category_id", -1)) == int(class_id)]
    n_gt = len(cls_gts)
    if n_gt == 0:
        return np.nan, 0, len(cls_preds)

    gts_by_img = {}
    for g in cls_gts:
        gts_by_img.setdefault(str(g["image_id"]), []).append({"bbox": g["bbox"], "matched": False})

    cls_preds = sorted(cls_preds, key=lambda x: float(x.get("score", 0.0)), reverse=True)
    tp = np.zeros(len(cls_preds), dtype=float)
    fp = np.zeros(len(cls_preds), dtype=float)

    for i, p in enumerate(cls_preds):
        img_id = str(p["image_id"])
        candidates = gts_by_img.get(img_id, [])
        best_iou = 0.0
        best_j = -1
        for j, g in enumerate(candidates):
            if g["matched"]:
                continue
            iou = box_iou(p["bbox"], g["bbox"])
            if iou > best_iou:
                best_iou = iou
                best_j = j
        if best_iou >= iou_thr and best_j >= 0:
            tp[i] = 1.0
            candidates[best_j]["matched"] = True
        else:
            fp[i] = 1.0

    if len(cls_preds) == 0:
        return 0.0, n_gt, 0

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recall = tp_cum / max(n_gt, 1)
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    ap = ap_from_precision_recall(precision, recall)
    return ap, n_gt, len(cls_preds)


def compute_per_class_map(preds, gts, num_classes=5, iou_thresholds=IOU_THRESHOLDS):
    rows = []
    for cls in range(num_classes):
        aps_by_thr = {}
        gt_count = 0
        pred_count = 0
        for thr in iou_thresholds:
            ap, n_gt, n_pred = compute_class_ap(preds, gts, cls, float(thr))
            aps_by_thr[float(thr)] = ap
            gt_count = n_gt
            pred_count = n_pred
        valid_aps = [v for v in aps_by_thr.values() if not pd.isna(v)]
        rows.append({
            "Class ID": cls,
            "Class": CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else str(cls),
            "GT objects": int(gt_count),
            "Pred objects": int(pred_count),
            "AP": float(np.mean(valid_aps)) if len(valid_aps) else np.nan,
            "AP50": aps_by_thr.get(0.50, np.nan),
            "AP75": aps_by_thr.get(0.75, np.nan),
        })
    return rows


def build_per_class_map_report(split="test"):
    rows = []
    for ratio in ["5%", "10%"]:
        for spec in RUN_SPECS:
            nb = spec["nb"]
            preds, gts, pred_name, _ = load_pred_gt(ratio, nb, split)
            if preds is None or gts is None:
                continue
            cls_rows = compute_per_class_map(preds, gts, num_classes=NUM_CLASSES)
            for item in cls_rows:
                item.update({
                    "Ratio": ratio,
                    "Notebook": nb,
                    "Model": spec["display"],
                    "Prediction source": pred_name,
                    "Split": split,
                })
                rows.append(item)
    return pd.DataFrame(rows)


per_class_map_df = build_per_class_map_report(split=PER_CLASS_MAP_SPLIT)
per_class_map_df.to_csv(OUTPUT_DIR / f"per_class_map_{PER_CLASS_MAP_SPLIT}.csv", index=False)

per_class_map_view = per_class_map_df[
    ["Ratio", "Model", "Class ID", "Class", "GT objects", "Pred objects", "AP", "AP50", "AP75", "Prediction source"]
].copy()


def bold_best_map_per_class(styler, df, numeric_cols=("AP", "AP50", "AP75")):
    """Bold best model value per (Ratio, Class ID) for each AP metric."""
    def highlight(col):
        styles = [""] * len(col)
        if col.name not in numeric_cols:
            return styles
        for (ratio, class_id), idx in df.groupby(["Ratio", "Class ID"]).groups.items():
            idx = list(idx)
            vals = pd.to_numeric(df.loc[idx, col.name], errors="coerce")
            if vals.notna().any():
                best = vals.max()
                for i in idx:
                    if pd.notna(df.loc[i, col.name]) and abs(float(df.loc[i, col.name]) - float(best)) < 1e-12:
                        styles[list(df.index).index(i)] = "font-weight: bold"
        return styles
    return styler.apply(highlight, axis=0)

styled_per_class_map = (
    per_class_map_view
    .style
    .format({"AP": "{:.3f}", "AP50": "{:.3f}", "AP75": "{:.3f}"})
    .pipe(bold_best_map_per_class, per_class_map_view, ("AP", "AP50", "AP75"))
)
display(styled_per_class_map)


def plot_grouped_per_class_map(metric="AP"):
    """Grouped bar chart: x = class 0–4, bars = models, for one AP metric."""
    for ratio in ["5%", "10%"]:
        sub = per_class_map_df[per_class_map_df["Ratio"] == ratio].copy()
        if sub.empty:
            continue
        models = [s["display"] for s in RUN_SPECS if s["display"] in set(sub["Model"])]
        class_ids = list(range(NUM_CLASSES))
        x = np.arange(len(class_ids))
        width = min(0.15, 0.80 / max(len(models), 1))
        fig, ax = plt.subplots(figsize=(13.5, 5.2))
        for i, model in enumerate(models):
            vals = []
            for cls in class_ids:
                row = sub[(sub["Model"] == model) & (sub["Class ID"] == cls)]
                vals.append(float(row[metric].iloc[0]) if len(row) else np.nan)
            offset = (i - (len(models) - 1) / 2) * width
            ax.bar(x + offset, vals, width=width, label=model, alpha=0.88)
            for xi, yi in zip(x + offset, vals):
                if pd.notna(yi):
                    ax.text(xi, yi + 0.01, f"{yi:.2f}", ha="center", va="bottom", fontsize=7, rotation=90)
        ax.set_title(f"{ratio} — per-class {metric} by model")
        ax.set_xlabel("Class")
        ax.set_ylabel(metric)
        ax.set_xticks(x)
        ax.set_xticklabels([str(c) for c in class_ids])
        ax.set_ylim(0, 1.08)
        ax.grid(axis="y", alpha=0.22)
        ax.legend(ncol=2, fontsize=8)
        savefig(f"per_class_{metric.lower()}_grouped_bar_{ratio.replace('%','pct')}.png", dpi=180)
        plt.show()


for metric in ["AP", "AP50", "AP75"]:
    plot_grouped_per_class_map(metric)


def plot_per_model_class_map_bars():
    """One compact figure per ratio. Each subplot is one model; bars are class 0–4 AP/AP50/AP75."""
    metrics_to_plot = ["AP", "AP50", "AP75"]
    for ratio in ["5%", "10%"]:
        sub_ratio = per_class_map_df[per_class_map_df["Ratio"] == ratio].copy()
        if sub_ratio.empty:
            continue
        fig, axes = plt.subplots(1, len(RUN_SPECS), figsize=(21, 4.2), sharey=True)
        if len(RUN_SPECS) == 1:
            axes = [axes]
        for ax, spec in zip(axes, RUN_SPECS):
            model_name = spec["display"]
            sub = sub_ratio[sub_ratio["Model"] == model_name].sort_values("Class ID")
            if sub.empty:
                ax.axis("off")
                continue
            x = np.arange(len(sub))
            width = 0.24
            for k, metric in enumerate(metrics_to_plot):
                vals = sub[metric].values.astype(float)
                ax.bar(x + (k - 1) * width, vals, width, label=metric, alpha=0.88)
            ax.set_title(model_name, fontsize=10)
            ax.set_xlabel("Class")
            ax.set_xticks(x)
            ax.set_xticklabels([str(int(cid)) for cid in sub["Class ID"]])
            ax.grid(axis="y", alpha=0.22)
            ax.set_ylim(0, 1.05)
        axes[0].set_ylabel("AP score")
        handles, labels = axes[-1].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
        fig.suptitle(f"{ratio} — class-wise AP/AP50/AP75 inside each model", y=1.16, fontsize=13)
        savefig(f"per_model_class_map_bars_{ratio.replace('%','pct')}.png", dpi=180)
        plt.show()

plot_per_model_class_map_bars()


## 9. Qualitative prediction grids

Phần này vẽ prediction giống kiểu YOLO comparison nhưng **không load `.pt` model**. Notebook dùng prediction JSON đã lưu sẵn.

Với mỗi ratio:
- 5 ảnh đầu là các ảnh có confidence dự đoán cao nhất.
- 5 ảnh sau là các ảnh có confidence dự đoán thấp nhất.
- Cột gồm: `Original`, `Ground Truth`, `Def-DETR Sup-only`, `Def-DETR SSOD`, `DETR-SSOD + SHM`, `SHM + CQC`, `Full SHM+CQC+CPM`.


In [ ]:
N_QUAL_IMAGES = 10
QUAL_CONF_THR = 0.25
QUAL_TOP_K = 5
QUAL_BOTTOM_K = 5

CLASS_COLORS = {
    0: "#d62728",
    1: "#2ca02c",
    2: "#1f77b4",
    3: "#ffbf00",
    4: "#9467bd",
}

def build_image_lookup(yolo_root):
    lookup = {}
    if yolo_root is None:
        return lookup
    for split in ["test", "valid", "train"]:
        d = Path(yolo_root) / split
        if not d.exists():
            continue
        for p in d.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                lookup[p.name] = p
                lookup[p.stem] = p
                lookup[str(p)] = p
    return lookup

IMAGE_LOOKUP = build_image_lookup(YOLO_ROOT)

def resolve_image_path(image_id, id_to_file=None):
    image_id = str(image_id)
    candidates = [image_id, Path(image_id).name, Path(image_id).stem]
    if id_to_file and image_id in id_to_file:
        fn = id_to_file[image_id]
        candidates += [fn, Path(fn).name, Path(fn).stem]
    for c in candidates:
        if c in IMAGE_LOOKUP and Path(IMAGE_LOOKUP[c]).exists():
            return Path(IMAGE_LOOKUP[c])
    return None

def draw_boxes(ax, boxes, with_scores=True, title=None, linewidth=1.8, fontsize=8):
    for item in boxes:
        x1, y1, x2, y2 = item["bbox"]
        cls = int(item.get("category_id", 0))
        score = item.get("score", None)
        color = CLASS_COLORS.get(cls, "#f28e2b")
        rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=linewidth)
        ax.add_patch(rect)
        label = str(cls)
        if with_scores and score is not None:
            label = f"{cls} {score:.2f}"
        ax.text(
            x1, max(0, y1 - 3), label,
            fontsize=fontsize, color="black",
            bbox=dict(facecolor="white", edgecolor=color, alpha=0.72, pad=0.7),
        )
    if title:
        ax.set_title(title, fontsize=10, pad=2)
    ax.axis("off")

def group_by_image(items):
    d = defaultdict(list)
    for x in items:
        d[str(x["image_id"])].append(x)
    return d

def load_preds_for_ratio(ratio, split="test"):
    all_preds, pred_sources = {}, {}
    for spec in RUN_SPECS:
        preds, _, pred_name, _ = load_pred_gt(ratio, spec["nb"], split)
        all_preds[spec["nb"]] = group_by_image(preds or [])
        pred_sources[spec["nb"]] = pred_name
    return all_preds, pred_sources

def choose_qual_images_by_prediction_score(ratio, top_k=5, bottom_k=5, conf_thr=0.25, selector_nb="NB06"):
    """Pick high/low confidence images using the selected final model, normally NB06."""
    nb02_dir = Path(summary_df[(summary_df["Ratio"] == ratio) & (summary_df["Notebook"] == "NB02")].iloc[0]["Run dir"])
    gt_path = nb02_dir / "gt_test_cache.json"
    gt_data = load_json(gt_path)
    gts = normalize_gt(gt_data)
    id_to_file = extract_image_id_to_file(gt_data)
    gt_by_img = group_by_image(gts)

    all_preds, _ = load_preds_for_ratio(ratio, "test")
    score_preds = all_preds.get(selector_nb, {}) or all_preds.get("NB05", {}) or all_preds.get("NB02", {})

    candidates = []
    for img_id, anns in gt_by_img.items():
        img_path = resolve_image_path(img_id, id_to_file)
        if img_path is None:
            continue
        preds = [p for p in score_preds.get(str(img_id), []) if p.get("score", 0.0) >= conf_thr]
        if preds:
            scores = [p.get("score", 0.0) for p in preds]
            mean_score = float(np.mean(scores))
            max_score = float(np.max(scores))
        else:
            mean_score = 0.0
            max_score = 0.0
        candidates.append({
            "image_id": str(img_id),
            "image_path": img_path,
            "num_gt": len(anns),
            "num_pred": len(preds),
            "mean_score": mean_score,
            "max_score": max_score,
        })

    if not candidates:
        return [], [], id_to_file, gt_by_img

    high = sorted(candidates, key=lambda x: (-x["mean_score"], -x["max_score"], -x["num_pred"], x["image_id"]))[:top_k]
    high_ids = {x["image_id"] for x in high}
    low_pool = [x for x in candidates if x["image_id"] not in high_ids]
    # Prefer images with at least GT so low-confidence rows are meaningful.
    low = sorted(low_pool, key=lambda x: (x["mean_score"], x["max_score"], -x["num_gt"], x["image_id"]))[:bottom_k]
    return high, low, id_to_file, gt_by_img

def plot_qualitative_subset(ratio, selected, id_to_file, gt_by_img, subset_name, conf_thr=0.25):
    if not selected:
        return

    all_preds, pred_sources = load_preds_for_ratio(ratio, "test")
    columns = ["Original", "Ground Truth"] + [spec["display"].replace("\\n", " ") for spec in RUN_SPECS]
    nrows, ncols = len(selected), len(columns)

    # Bigger cells, very small spacing. Saved image is large enough for report zooming.
    fig_w = 4.85 * ncols
    fig_h = 4.65 * nrows
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h), squeeze=False)

    for row_idx, item in enumerate(selected):
        img_id = item["image_id"]
        img_path = item["image_path"]
        img = Image.open(img_path).convert("RGB")
        score_text = f"mean={item['mean_score']:.2f}  max={item['max_score']:.2f}  n={item['num_pred']}"

        ax = axes[row_idx, 0]
        ax.imshow(img, cmap="gray")
        ax.set_title("Original" if row_idx == 0 else "", fontsize=10, pad=2)
        ax.axis("off")
        ax.text(
            0.01, 0.99, f"{Path(img_path).name}\n{score_text}",
            transform=ax.transAxes,
            va="top", ha="left", fontsize=7.5,
            bbox=dict(facecolor="white", alpha=0.70, edgecolor="none", pad=0.5)
        )

        ax = axes[row_idx, 1]
        ax.imshow(img, cmap="gray")
        draw_boxes(
            ax,
            gt_by_img.get(str(img_id), []),
            with_scores=False,
            title=("Ground Truth" if row_idx == 0 else None),
            linewidth=1.7,
            fontsize=7.5
        )

        for col_offset, spec in enumerate(RUN_SPECS, start=2):
            nb = spec["nb"]
            ax = axes[row_idx, col_offset]
            ax.imshow(img, cmap="gray")
            preds = [p for p in all_preds.get(nb, {}).get(str(img_id), []) if p.get("score", 1.0) >= conf_thr]
            title = spec["display"].replace("\\n", " ") if row_idx == 0 else None
            draw_boxes(ax, preds, with_scores=True, title=title, linewidth=1.7, fontsize=7.5)

    fig.suptitle(f"{subset_name} predictions — ratio {ratio} — conf ≥ {conf_thr}", fontsize=14, y=0.996)
    fig.subplots_adjust(left=0.002, right=0.998, top=0.975, bottom=0.002, wspace=0.006, hspace=0.018)
    out = OUTPUT_DIR / f"qualitative_predictions_{ratio.replace('%','pct')}_{subset_name.lower().replace(' ', '_')}.png"
    plt.savefig(out, dpi=190, bbox_inches="tight", pad_inches=0.03)
    print(f"saved: {out.name}")
    plt.show()

def plot_qualitative_grid(ratio, top_k=5, bottom_k=5, conf_thr=0.25):
    high, low, id_to_file, gt_by_img = choose_qual_images_by_prediction_score(
        ratio, top_k=top_k, bottom_k=bottom_k, conf_thr=conf_thr, selector_nb="NB06"
    )
    if not high and not low:
        print(f"no qualitative images found for {ratio}")
        return
    plot_qualitative_subset(ratio, high, id_to_file, gt_by_img, "High-confidence", conf_thr=conf_thr)
    plot_qualitative_subset(ratio, low, id_to_file, gt_by_img, "Low-confidence", conf_thr=conf_thr)

for ratio in ["5%", "10%"]:
    plot_qualitative_grid(ratio, top_k=QUAL_TOP_K, bottom_k=QUAL_BOTTOM_K, conf_thr=QUAL_CONF_THR)

## 9. Final interpretation printout

In [ ]:
interpret_rows = []
for ratio in ["5%", "10%"]:
    sub = summary_df[summary_df["Ratio"] == ratio].sort_values("nb_order")
    if len(sub) == 0:
        continue
    base = float(sub.iloc[0]["Test AP"])
    best = sub.sort_values("Test AP", ascending=False).iloc[0]
    for _, r in sub.iterrows():
        interpret_rows.append({
            "Ratio": ratio,
            "Method": r["Display"],
            "Val AP": float(r["Val AP"]),
            "Test AP": float(r["Test AP"]),
            "Δ Test AP vs Sup-only": float(r["Test AP"] - base),
        })
    print(f"{ratio}: best test AP = {best['Test AP']:.4f} ({best['Display']})")

interpret_df = pd.DataFrame(interpret_rows)
display(interpret_df)

interpret_df.to_csv(OUTPUT_DIR / "summary_interpretation_table.csv", index=False)
print(f"summary outputs saved to: {OUTPUT_DIR}")